# TEST-104 OFFICIAL — chạm **LẦN THỨ HAI** · UniFormer-S ensemble 5 fold

> **Research Use Only (RUO)** — chưa kiểm định lâm sàng, không dùng chẩn đoán.

⚠️⚠️ **Tập test này ĐÃ BỊ NHÌN MỘT LẦN** (2026-08-07, cấu hình cũ, macro-F1 0.6162).
Mọi báo cáo dùng số của lần chạy này **bắt buộc** nói rõ đây là lần chạm thứ hai.

Mọi lựa chọn đã khoá ở **`docs/TEST104_PREREGISTRATION.md` §B**, commit trước khi chạy.
**Không có tham số nào trong notebook này để sửa.** Sửa gì cũng là phá pre-registration,
và con số ra sẽ không dùng được để báo cáo.

## Cần mount gì

| | |
|---|---|
| **Cache lưới `128×128×16`** | sinh bởi `configs/preprocess_cghnet.yaml`, phải đủ **498** ca (394 trainval + 104 test) |
| **5 checkpoint** | sha256 đã ghim trong pre-registration §B1 |
| Internet | **KHÔNG cần** — không tải trọng số pretrained, checkpoint đã có sẵn |

**Bố cục thư mục checkpoint không quan trọng.** Notebook dò bằng **sha256**, không bằng
tên file hay tên thư mục: mỗi file `.pt` dưới `/kaggle/input` được băm rồi đối chiếu với
bộ mã đã ghim trong pre-registration. Một file khớp `pinned[3]` **là** checkpoint của
fold 3, bất kể nó nằm ở đâu và tên là gì.

Cách này chặt hơn dò theo đường dẫn chứ không lỏng hơn: tên thư mục là thứ người ta đặt
tay và đặt sai được, còn mã băm thì không.

## Notebook này làm gì, và cố ý KHÔNG làm gì

Nó **chỉ suy luận và lưu xác suất**, không in một metric nào. Đọc số là việc của
`src.eval.test_report`, chạy trên CPU từ file `.npz` đã lưu.

Tách hai bước là có chủ đích: bước chạm test xảy ra **một lần** và tốn GPU. Gộp phần báo
cáo vào đây thì mỗi lần muốn xem thêm một metric lại phải chạy lại suy luận — mà "chạy
lại" trên test-104 chính là thứ bị cấm.

Ngoại lệ duy nhất được in ngay: **latency**. Nó chỉ đo được trong lúc chạy, và lần chạm
trước đã bỏ lỡ rồi không truy lại được.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"

# ---- KHÔNG CÓ THAM SỐ NÀO ĐỂ SỬA -------------------------------------------
FOLDS = [1, 2, 3, 4, 5]
CONFIG_NAME = "uniformer_s.yaml"
PIN_SET = "uniformer"          # bộ sha256 đã ghim trong pre-registration §B1
# ----------------------------------------------------------------------------

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
# Clone ĐẦY ĐỦ, không --depth 1: cổng pre-registration kiểm bằng `git log` trên file đó,
# và một clone nông có thể không mang theo commit chứa nó.
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_OUTPUT_DIR"] = "/kaggle/working/runs/test104_uniformer"
os.environ.pop("LLDMMRI_DATA_ROOT", None)

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"

CFG_PATH = REPO / "configs" / CONFIG_NAME
CFG = load_yaml(CFG_PATH)
PRE = load_yaml(REPO / "configs" / "preprocess_cghnet.yaml")
print(f"config: {CFG_PATH.name} · pin-set: {PIN_SET} · folds {FOLDS}")

## Cổng 0 ⚠️⚠️ — pre-registration §B đã commit chưa

Cổng này chạy **trước mọi thứ khác**. Một pre-registration viết sau khi nhìn số thì vô
nghĩa, nên nó được kiểm bằng `git log` chứ không bằng sự tồn tại của file.

`test_once` tự kiểm lại điều này lúc chạy; cell đây chỉ để bạn biết **ngay bây giờ** thay
vì sau khi đã mount xong mọi thứ.

In [ ]:
import subprocess as _sp

PREREG = "docs/TEST104_PREREGISTRATION.md"
_log = _sp.run(["git", "log", "-1", "--format=%h %ad %s", "--date=short", "--", PREREG],
               cwd=REPO, capture_output=True, text=True).stdout.strip()
_dirty = _sp.run(["git", "status", "--porcelain", "--", PREREG],
                 cwd=REPO, capture_output=True, text=True).stdout.strip()

assert _log, f"⛔ {PREREG} CHƯA COMMIT — dừng. Commit và push trước khi chạy."
assert not _dirty, f"⛔ {PREREG} có thay đổi chưa commit: {_dirty!r}"

_noi_dung = (REPO / PREREG).read_text("utf-8")
assert "§B" in _noi_dung, "⛔ pre-registration chưa có mục §B cho lần chạm thứ hai"
for _khoa in ("uniformer_s.yaml", "62948396cdccd5a4", "8edf4fbc07f181b2"):
    assert _khoa in _noi_dung, f"⛔ §B không nhắc {_khoa} — sai bản pre-registration"

print("pre-registration commit:", _log)
print("cổng 0 ✓ — §B đã commit và sạch")

## 1. Cache và checkpoint

Checkpoint được nhận diện bằng **sha256**, không bằng đường dẫn — xem phần đầu notebook.
Cell in ra cây thư mục đã mount trước, để nếu có gì thiếu thì bạn thấy ngay *cái gì* thiếu
thay vì chỉ thấy một `assert` đỏ.

In [ ]:
import os as _os

INPUT_ROOT = Path("/kaggle/input")

print("đã mount:")
for d in sorted(INPUT_ROOT.glob("*")):
    n_npz = len(list(d.rglob("*.npz")))
    n_pt = len(list(d.rglob("*.pt")))
    print(f"  {d.name:<40} {n_npz:>5} .npz  {n_pt:>3} .pt")

# --- cache: thư mục có cache_meta.json và đủ .npz ---
CACHE_DIR = None
for cand in sorted(INPUT_ROOT.rglob("cache_meta.json")):
    if len(list(cand.parent.glob("*.npz"))) > 400:
        CACHE_DIR = cand.parent
        break
assert CACHE_DIR, (
    "⛔ không thấy cache đã mount. Cần Dataset chứa cache_meta.json + ~498 file .npz "
    "sinh bởi configs/preprocess_cghnet.yaml."
)
os.environ["LLDMMRI_CACHE_DIR"] = str(CACHE_DIR)
print("\ncache:", CACHE_DIR)

## Cổng A ⚠️⚠️ — cache có ĐÚNG hình học mà model được train không

Cache sai hình học là chế độ hỏng **im lặng** tệ nhất ở đây: nếu lưới khác mà kích thước
vẫn hợp lệ, model chạy trơn, ra 104 xác suất trông hoàn toàn bình thường, và con số sai
sẽ được báo cáo như thật. Trên một tập chạm một lần thì không có cơ hội sửa.

Cổng đối chiếu `cache_meta.json` với **chính file config** trong repo, không với một bản
chép tay — hai bản chép sẽ trôi khỏi nhau.

In [ ]:
import json

meta = json.loads((CACHE_DIR / "cache_meta.json").read_text("utf-8"))

# Đọc kỳ vọng TỪ config trong repo, không ghi cứng ở đây.
CAN = {
    "axis_order": PRE["axis_order"],
    "align_phases": PRE["align_phases"],
    "reference_phase": PRE["reference_phase"],
    "crop_mode": PRE["crop_mode"],
    "target_spacing": PRE["target_spacing"],
    "target_size": PRE["target_size"],
    "crop_margin_voxels": PRE["crop_margin_voxels"],
}
for key, want in CAN.items():
    got = meta.get(key)
    assert got == want, f"⛔ cache SAI: {key} = {got!r}, cần {want!r}"
assert meta["lesion_tight"]["source"] == "mask", "⛔ phải cắt theo mask, không phải bbox"

# Model nhận đúng khối này; và crop_size của config train phải khớp target_size của cache.
assert list(CFG["data"]["crop_size"]) == list(PRE["target_size"]), (
    f"⛔ config train cắt {CFG['data']['crop_size']} nhưng cache dựng cho {PRE['target_size']}"
)

n_npz = len(list(CACHE_DIR.glob("*.npz")))
assert n_npz >= 498, (
    f"⛔ chỉ có {n_npz} ca, cần 498. Cache thiếu test-104 — build_cache phải chạy trên "
    "toàn bộ annotation, không riêng trainval."
)
print(f"cổng A ✓ · hình học khớp config · {n_npz} ca · cache commit {meta.get('git_commit')}")

## Cổng B ⚠️⚠️ — checkpoint có đúng bộ đã ghim không

Băm **mọi** file `.pt` đã mount rồi đối chiếu với bộ mã trong pre-registration §B1. File
nào khớp `pinned[N]` chính là checkpoint của fold `N` — đây là bằng chứng mạnh hơn tên
thư mục, vì tên là thứ đặt tay và đặt sai được.

Sau khi khớp, cell dựng một cây chuẩn `fold_N/best.pt` bằng symlink trong `/kaggle/working`
để bước chạm nhận được bố cục nó mong đợi. **`test_once` sẽ băm lại lần nữa** và tự từ chối
nếu lệch — hai lớp kiểm độc lập, không phải một lớp lặp lại.

Cổng cũng chặn hai fold cùng trỏ về một file: "ensemble" đếm một model hai lần vẫn cho ra
con số trông hợp lý, nên về sau không có dấu hiệu nào để phát hiện.

In [ ]:
from src.eval.test_once import PIN_SETS, find_checkpoints, sha16

pinned = PIN_SETS[PIN_SET]
nguoc = {sha: f for f, sha in pinned.items()}
assert len(nguoc) == len(pinned), "⛔ bộ ghim có mã trùng nhau"

CKPTS, la = {}, []
for p in sorted(INPUT_ROOT.rglob("*.pt")):
    d = sha16(p)
    if d in nguoc:
        f = nguoc[d]
        assert f not in CKPTS, f"⛔ hai file cùng khớp fold {f}: {CKPTS[f]} và {p}"
        CKPTS[f] = p
    else:
        la.append(p)

print(f"{'fold':<6}{'sha256':<20}file")
for f in FOLDS:
    if f in CKPTS:
        print(f"{f:<6}{pinned[f]:<20}{CKPTS[f]}")
    else:
        print(f"{f:<6}{pinned[f]:<20}⛔ KHÔNG THẤY")

if la:
    print(f"\n{len(la)} file .pt không khớp bộ ghim (bỏ qua, không phải lỗi):")
    for p in la[:5]:
        print(f"  {p}  sha {sha16(p)}")

thieu = [f for f in FOLDS if f not in CKPTS]
assert not thieu, (
    f"⛔ thiếu checkpoint cho fold {thieu}. Không file .pt nào đã mount có sha256 khớp bộ "
    f"ghim {PIN_SET!r} trong pre-registration §B1. Kiểm lại Dataset đã add vào notebook "
    "chưa, và có đúng là bộ checkpoint đã khoá không."
)
assert len({p.resolve() for p in CKPTS.values()}) == len(FOLDS), "⛔ hai fold trỏ cùng một file"

# Dựng cây chuẩn để `test_once.find_checkpoints` nhận ra. Symlink, không copy: file gốc
# ở /kaggle/input là chỉ-đọc và không bị đụng, mà cũng không tốn thêm 425 MB đĩa.
import shutil as _shutil

CKPT_DIR = Path("/kaggle/working/ckpt_uniformer")
_shutil.rmtree(CKPT_DIR, ignore_errors=True)
for f in FOLDS:
    d = CKPT_DIR / f"fold_{f}"
    d.mkdir(parents=True, exist_ok=True)
    _os.symlink(CKPTS[f].resolve(), d / "best.pt")

lai = find_checkpoints(CKPT_DIR, FOLDS)
assert len(lai) == len(FOLDS), f"⛔ cây chuẩn dựng hỏng: {sorted(lai)}"
assert all(sha16(lai[f]) == pinned[f] for f in FOLDS), "⛔ symlink trỏ sai file"

print(f"\ncổng B ✓ — đủ 5 checkpoint đã khoá, không cái nào trùng")
print("cây chuẩn:", CKPT_DIR)

## 2. Chạm test-104

**Cell này là lần chạm.** Nó tự kiểm lại bốn thứ và **nổ** chứ không cảnh báo:

1. pre-registration đã commit (kiểm bằng `git log`)
2. `Splits.validate()` — trong đó có `val_fold_i ∩ test = ∅` với mọi `i`
3. sha256 khớp bộ đã ghim `uniformer`
4. không có hai checkpoint trùng nhau

104 ca × 5 model, khoảng một tới hai phút.

⚠️ **Không in metric ở đây, có chủ đích.** Xem phần đầu notebook.

In [ ]:
from src.eval.test_once import run as touch_test

print("checkpoint dir:", CKPT_DIR, "(cây chuẩn dựng ở cổng B)")
print("cache dir:     ", CACHE_DIR)
print("pin-set:       ", PIN_SET)

path = touch_test(
    ckpt_dir=CKPT_DIR,
    config_path=CFG_PATH,
    out_dir=os.environ["LLDMMRI_OUTPUT_DIR"],
    cache_dir=CACHE_DIR,
    folds=FOLDS,
    pin_set=PIN_SET,
)
print("\nđã lưu:", path)

## 3. Latency — con số DUY NHẤT được đọc ngay tại đây

Nó chỉ đo được trong lúc chạy. Lần chạm trước có sẵn con số này miễn phí, không ghi lại,
và sau đó không truy ra được nữa — test chạm một lần nên không chạy lại để đo được.

⚠️ **Đây là latency theo LÔ trên T4, không phải độ trễ một ca đơn lẻ.** Web app phục vụ
từng ca một nên sẽ chậm hơn con số này. Báo cáo phải nói rõ điều đó, đừng trình bày
`per_case` như thời gian đáp ứng của hệ thống thật.

In [ ]:
meta_run = json.loads((Path(path).parent / "test_run_meta.json").read_text("utf-8"))
lat = meta_run["latency"]

print(f"thiết bị        : {lat['device']} · batch {lat['batch_size']} · amp={lat['amp']}")
print(f"số ca           : {lat['n_cases']}")
print(f"giây mỗi model  : {lat['seconds_per_member']}")
print()
print(f"  1 model      : {lat['per_case_1model_ms']:>8.1f} ms/ca   (so được với văn liệu)")
print(f"  ensemble {len(lat['seconds_per_member'])}   : {lat['per_case_ensemble_ms']:>8.1f} ms/ca   "
      "(thứ hệ thống thật phải trả)")
print()
print("⚠ Đo theo lô. Một ca đơn lẻ sẽ chậm hơn — không có lợi thế theo lô, và cộng thêm")
print("  thời gian đọc + tiền xử lý NIfTI mà con số này KHÔNG bao gồm.")

## 4. Gói mang về

In [ ]:
import shutil

OUT_ROOT = Path(os.environ["LLDMMRI_OUTPUT_DIR"])
PACK = Path("/kaggle/working/test104_uniformer_results")
shutil.rmtree(PACK, ignore_errors=True)
PACK.mkdir(parents=True)

for name in ("test_probs.npz", "test_run_meta.json"):
    src = OUT_ROOT / name
    assert src.exists(), f"⛔ thiếu {name}"
    shutil.copy2(src, PACK / name)

for f in sorted(PACK.rglob("*")):
    print(f"  {f.relative_to(PACK)}  {f.stat().st_size / 2**10:.1f} KiB")

print()
print((PACK / "test_run_meta.json").read_text("utf-8"))

print("""
⚠ TẢI VỀ: giải nén CHỈ MỘT LỚP. File .npz bản thân là zip; trình giải nén bung đệ quy sẽ
  biến nó thành thư mục và src.eval.* sẽ không thấy.

Đặt vào runs/test104_uniformer/ rồi chạy ở local (CPU, chạy lại bao nhiêu lần cũng được,
KHÔNG thành lần chạm thứ ba):

    python -m src.eval.test_report --run-dir runs/test104_uniformer \
        --oof-dir runs/Uniformer3D

  ⚠ --oof-dir BẮT BUỘC và phải là run out-of-fold của CHÍNH cấu hình này. `T` được fit ở
    đó rồi áp mù lên test. Trỏ nhầm sang run khác là fit T trên một phân bố xác suất khác
    — sai im lặng, không nổ.

Và phép so ghép cặp với lần chạm 1 trên đúng 104 ca đó — pre-registration §B6:

    python -m src.eval.compare --baseline runs/test104 --candidate runs/test104_uniformer
""")